In [1]:
# simu_matrix.ipynb - add CNA signals to adata count data.

In [2]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import warnings
from sccnasim.cs.core import cs_cna_core
from sccnasim.cs.pp import calc_size_factors
from sccnasim.cs.main import sample_barcodes
from sccnasim.xlib.xrange import format_chrom
from scipy.sparse import csr_matrix

# Load count matrix as adata

In [3]:
work_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/scdesign2_screadsim/simu"
sample_prefix = 'HCC3N_simu.intergene'

# the two 'gene' and 'intergene' should have the same barcodes.
#simu_barcode_fn = None
simu_barcode_fn = os.path.join(work_dir, 'matrix/barcodes.simu.tsv')    # default None

In [4]:
in_matrix_fn = os.path.join(work_dir, 'matrix/%s.countmatrix.txt' % sample_prefix)
in_barcode_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/scdesign2_screadsim/data/barcodes.tsv'
barcode_whitelist_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/base/data/refapp/cellranger/737K-august-2016.txt'

In [5]:
gene_anno = pd.read_csv(in_matrix_fn, header = None, delimiter = '\t', usecols = [0])
gene_anno.columns = ['gene']
gene_anno

,gene
0,1_0_29554
1,1_31109_34554
2,1_36081_65419
3,1_71585_89295
4,1_133723_139790
...,...
21210,9_137217009_137217452
21211,9_137219361_137220247
21212,9_137221581_137224635
21213,9_137226311_137227271


In [6]:
gene_anno[['chrom', 'start', 'end']] = gene_anno['gene'].str.split('_', expand = True)
gene_anno['start'] = gene_anno['start'].astype(int)
gene_anno['end'] = gene_anno['end'].astype(int)
gene_anno

,gene,chrom,start,end
0,1_0_29554,1,0,29554
1,1_31109_34554,1,31109,34554
2,1_36081_65419,1,36081,65419
3,1_71585_89295,1,71585,89295
4,1_133723_139790,1,133723,139790
...,...,...,...,...
21210,9_137217009_137217452,9,137217009,137217452
21211,9_137219361_137220247,9,137219361,137220247
21212,9_137221581_137224635,9,137221581,137224635
21213,9_137226311_137227271,9,137226311,137227271


In [7]:
cell_anno = pd.read_csv(in_barcode_fn, sep = '\t', header = None)
cell_anno.columns = ['cell']
cell_anno

,cell
0,AAACAAGTATCTCCCA-1
1,AAACATTTCCCGGATT-1
2,AAACCGTTCGTCCAGG-1
3,AAACCTAAGCAGCCGG-1
4,AAACGGGCGTACGGGT-1
...,...
595,TTGTGCAGCCACGTCA-1
596,TTGTGGTGGTACTAAG-1
597,TTGTGTATGCCACCAA-1
598,TTGTGTTTCCCGAAAG-1


In [8]:
cell_anno['cell_type'] = 'normal'

In [9]:
data = np.genfromtxt(in_matrix_fn, delimiter = '\t', usecols = range(1, 601))
data.shape

(21215, 600)

In [10]:
adata = ad.AnnData(
    X = data.T,
    obs = cell_anno,
    var = gene_anno
)
adata

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 600 × 21215
    obs: 'cell', 'cell_type'
    var: 'gene', 'chrom', 'start', 'end'

In [11]:
target_chroms = [str(i) for i in range(1, 23)]
adata = adata[:, adata.var['chrom'].isin(target_chroms)].copy()
adata

AnnData object with n_obs × n_vars = 600 × 21215
    obs: 'cell', 'cell_type'
    var: 'gene', 'chrom', 'start', 'end'

In [12]:
adata.var['chrom'].value_counts()

chrom
1     2208
2     1554
11    1233
19    1222
17    1130
6     1107
3     1069
7     1065
5     1056
12    1048
4      971
14     952
16     887
8      875
9      874
10     857
15     663
20     623
22     573
13     476
18     420
21     352
Name: count, dtype: int64

In [13]:
adata.write_h5ad(os.path.join(work_dir, 'matrix/%s.raw.h5ad' % sample_prefix), compression = 'gzip')

# Find CNA genes

In [14]:
cna_profile_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/data/cna_profile.tsv"
out_dir = os.path.join(work_dir, 'matrix')

## Load data

In [15]:
#adata.X = adata.X.astype('float32')

In [16]:
adata.var['chrom'] = adata.var['chrom'].astype(str)
adata.var

,gene,chrom,start,end
0,1_0_29554,1,0,29554
1,1_31109_34554,1,31109,34554
2,1_36081_65419,1,36081,65419
3,1_71585_89295,1,71585,89295
4,1_133723_139790,1,133723,139790
...,...,...,...,...
21210,9_137217009_137217452,9,137217009,137217452
21211,9_137219361_137220247,9,137219361,137220247
21212,9_137221581_137224635,9,137221581,137224635
21213,9_137226311_137227271,9,137226311,137227271


In [17]:
cna_profile = pd.read_csv(cna_profile_fn, sep = '\t', header = None, dtype = {0:str})
cna_profile.columns = ['chrom', 'start', 'end', 'clone', 'cn0', 'cn1']
cna_profile

,chrom,start,end,clone,cn0,cn1
0,chr1,123400001,248956422,tumor,1,2
1,chr4,50000001,190214555,tumor,0,1
2,chr8,1,45200000,tumor,0,1
3,chr8,45200001,145138636,tumor,1,2
4,chr13,17700001,114364328,tumor,2,0
5,chr17,1,25100000,tumor,2,0


In [18]:
cna_profile['region'] = cna_profile['chrom'] + ':' + cna_profile['start'].map(str) + '-' + cna_profile['end'].map(str)
cna_profile

cna_profile['cna_type'] = 'N.A.'
cna_profile.loc[cna_profile['cn0'] + cna_profile['cn1'] > 2, 'cna_type'] = 'gain'
cna_profile.loc[cna_profile['cn0'] + cna_profile['cn1'] < 2, 'cna_type'] = 'loss'
cna_profile.loc[cna_profile['cn0'] + cna_profile['cn1'] == 2, 'cna_type'] = 'loh'
cna_profile

,chrom,start,end,clone,cn0,cn1,region,cna_type
0,chr1,123400001,248956422,tumor,1,2,chr1:123400001-248956422,gain
1,chr4,50000001,190214555,tumor,0,1,chr4:50000001-190214555,loss
2,chr8,1,45200000,tumor,0,1,chr8:1-45200000,loss
3,chr8,45200001,145138636,tumor,1,2,chr8:45200001-145138636,gain
4,chr13,17700001,114364328,tumor,2,0,chr13:17700001-114364328,loh
5,chr17,1,25100000,tumor,2,0,chr17:1-25100000,loh


In [19]:
def get_overlap_genes(df, anno):
    df["chrom"] = df["chrom"].map(format_chrom)
    anno["chrom"] = anno["chrom"].map(format_chrom)

    df = df.drop_duplicates("region", ignore_index = True)
    anno = anno.drop_duplicates("gene", ignore_index = True)

    overlap = df.groupby("region").apply(
        lambda x: anno.loc[
            (anno["chrom"] == x["chrom"].values[0]) &
            (anno["start"] <= x["end"].values[0]) &
            (anno["end"] >= x["start"].values[0])
        ]).reset_index()
    
    if 'level_1' in overlap.columns:
        overlap.drop(columns = ['level_1'], inplace = True)
    overlap = overlap.merge(df[['region', 'cn0', 'cn1', 'cna_type']], on = 'region', how = 'left')

    stat = overlap.groupby("gene").size().reset_index(name = "n_overlap_region")
    dup = stat[stat["n_overlap_region"] > 1]
    uniq = stat[stat["n_overlap_region"] == 1].merge(overlap, on = "gene", how = "left")
    uniq = uniq.sort_values(by = ["region", 'chrom', 'start', 'end', 'gene'])

    res = dict(
        # overlap : pandas.DataFrame
        #   The overlapping results. It contains two columns:
        #   - "region": region ID.
        #   - "gene": name of genes overlapping the region.
        overlap = overlap,

        # n_region : int
        #   Number of unique regions.
        n_region = df.shape[0],

        # n_region_overlap : int
        #   Number of unique regions that have overlapping genes.
        n_region_overlap = len(overlap["region"].unique()),

        # n_gene : int
        #   Number of unique genes.
        n_gene = anno.shape[0],

        # n_gene_overlap : int
        #   Number of unique genes that have overlapping regions.
        n_gene_overlap = len(overlap["gene"].unique()),

        # n_gene_dup : int
        #   Number of genes overlapping more than 1 regions.
        n_gene_dup = dup.shape[0],

        # overlap_uniq : pandas.DataFrame
        #   The overlapping results.
        #   Similar to `overlap`, but the genes overlapping more than 1 regions
        #   are removed.
        overlap_uniq = uniq
    )

    return(res)

In [20]:
res = get_overlap_genes(df = cna_profile, anno = adata.var.copy())

/tmp/pbs.1793180.xomics/ipykernel_57707/1258343097.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  overlap = df.groupby("region").apply(


In [21]:
res.keys()

dict_keys(['overlap', 'n_region', 'n_region_overlap', 'n_gene', 'n_gene_overlap', 'n_gene_dup', 'overlap_uniq'])

In [22]:
res['overlap']

,region,gene,chrom,start,end,cn0,cn1,cna_type
0,chr13:17700001-114364328,13_0_18195297,13,0,18195297,2,0,loh
1,chr13:17700001-114364328,13_18232024_18540497,13,18232024,18540497,2,0,loh
2,chr13:17700001-114364328,13_18550697_18610298,13,18550697,18610298,2,0,loh
3,chr13:17700001-114364328,13_18611675_18672827,13,18611675,18672827,2,0,loh
4,chr13:17700001-114364328,13_18676163_18905439,13,18676163,18905439,2,0,loh
...,...,...,...,...,...,...,...,...
3409,chr8:45200001-145138636,8_144902168_144930358,8,144902168,144930358,1,2,gain
3410,chr8:45200001-145138636,8_144950888_144992914,8,144950888,144992914,1,2,gain
3411,chr8:45200001-145138636,8_144994837_145002811,8,144994837,145002811,1,2,gain
3412,chr8:45200001-145138636,8_145006046_145052378,8,145006046,145052378,1,2,gain


In [23]:
res['overlap_uniq']

,gene,n_overlap_region,region,chrom,start,end,cn0,cn1,cna_type
0,13_0_18195297,1,chr13:17700001-114364328,13,0,18195297,2,0,loh
93,13_18232024_18540497,1,chr13:17700001-114364328,13,18232024,18540497,2,0,loh
94,13_18550697_18610298,1,chr13:17700001-114364328,13,18550697,18610298,2,0,loh
95,13_18611675_18672827,1,chr13:17700001-114364328,13,18611675,18672827,2,0,loh
96,13_18676163_18905439,1,chr13:17700001-114364328,13,18676163,18905439,2,0,loh
...,...,...,...,...,...,...,...,...,...
2853,8_144902168_144930358,1,chr8:45200001-145138636,8,144902168,144930358,1,2,gain
2854,8_144950888_144992914,1,chr8:45200001-145138636,8,144950888,144992914,1,2,gain
2855,8_144994837_145002811,1,chr8:45200001-145138636,8,144994837,145002811,1,2,gain
2856,8_145006046_145052378,1,chr8:45200001-145138636,8,145006046,145052378,1,2,gain


In [24]:
res['overlap'].loc[~(res['overlap']['gene'].isin(res['overlap_uniq']['gene']))]

,region,gene,chrom,start,end,cn0,cn1,cna_type
2855,chr8:1-45200000,8_43202855_46822237,8,43202855,46822237,0,1,loss
2856,chr8:45200001-145138636,8_43202855_46822237,8,43202855,46822237,1,2,gain


In [25]:
res['overlap_uniq']['cna_type'].value_counts()

cna_type
gain    1609
loss    1005
loh      798
Name: count, dtype: int64

In [26]:
res['overlap'].to_csv(
    os.path.join(out_dir, 'cna_genes.tsv'),
    sep = '\t',
    index = False
)

In [27]:
res['overlap_uniq'].to_csv(
    os.path.join(out_dir, 'cna_genes.unique.tsv'),
    sep = '\t',
    index = False
)

### Update adata, masking multi-overlap features

In [28]:
mof = res['overlap'].loc[~(res['overlap']['gene'].isin(res['overlap_uniq']['gene'])), 'gene'].to_numpy()
mof

array(['8_43202855_46822237', '8_43202855_46822237'], dtype=object)

In [29]:
adata.X[:, adata.var['gene'].isin(mof)] = 0

### CN ratio

In [30]:
adata.var = adata.var.merge(res['overlap_uniq'][['gene', 'cna_type']], on = 'gene', how = 'left')
adata.var

,gene,chrom,start,end,cna_type
0,1_0_29554,1,0,29554,NaN
1,1_31109_34554,1,31109,34554,NaN
2,1_36081_65419,1,36081,65419,NaN
3,1_71585_89295,1,71585,89295,NaN
4,1_133723_139790,1,133723,139790,NaN
...,...,...,...,...,...
21210,9_137217009_137217452,9,137217009,137217452,NaN
21211,9_137219361_137220247,9,137219361,137220247,NaN
21212,9_137221581_137224635,9,137221581,137224635,NaN
21213,9_137226311_137227271,9,137226311,137227271,NaN


In [31]:
adata.var.loc[adata.var['cna_type'].isna(), 'cna_type'] = 'neutral'
adata.var['cna_type'].value_counts()

cna_type
neutral    17803
gain        1609
loss        1005
loh          798
Name: count, dtype: int64

In [32]:
adata.var['feature'] = adata.var['gene']

In [33]:
adata_fn = os.path.join(work_dir, 'matrix/%s.overlap_uniq_filter.h5ad' % sample_prefix)
adata.write_h5ad(adata_fn, compression = 'gzip')

In [34]:
df = adata.var.copy()
df['cn_ratio'] = 1.0
df.loc[df['cna_type'] == 'gain', 'cn_ratio'] = 1.5
df.loc[df['cna_type'] == 'loss', 'cn_ratio'] = 0.5
df['cn_ratio'].value_counts()

cn_ratio
1.0    18601
1.5     1609
0.5     1005
Name: count, dtype: int64

In [35]:
cn_ratio = {'tumor':df['cn_ratio'].to_numpy()}

# Simulate new CNA matrix

In [36]:
adata.X = csr_matrix(adata.X)

In [37]:
size_factors_train, size_factors_simu = calc_size_factors(
    adata = adata,
    size_factor = 'libsize',
    clone_cell_types = np.array(['normal', 'normal']),
    n_cell_each = [600, 600],
    kwargs_fit_sf = {"dist": "lognormal"},
    verbose = True
)

In [38]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    adata_simu = cs_cna_core(
        count_fn = adata_fn,
        out_dir = out_dir,
        out_prefix = sample_prefix,
        clones = pd.Series(['normal', 'tumor']),
        cell_types = pd.Series(['normal', 'normal']),
        n_cell_each = [600, 600],
        cn_ratio = cn_ratio,
        size_factor = 'libsize',
        size_factors_train = size_factors_train,
        size_factors_simu = size_factors_simu,
        marginal = 'auto',
        kwargs_fit_rd = {"min_nonzero_num": 3, "max_iter": 1000, "pval_cutoff": 0.05},
        libsize_ratio = 1.0,
        ncores = 10,
        verbose = True
    )

In [39]:
adata_simu

AnnData object with n_obs × n_vars = 1200 × 21215
    obs: 'cell_type'
    var: 'feature'

## Save adata

In [40]:
simu_barcode_fn = os.path.join(out_dir, 'barcodes.simu.tsv')
if simu_barcode_fn is None:
    barcodes = sample_barcodes(
        fn = barcode_whitelist_fn,
        n = adata_simu.shape[0],
        suffix = "-1",
        sort = True
    )
    adata_simu.obs["cell"] = barcodes
    fn = os.path.join(out_dir, 'barcodes.simu.tsv')
    adata_simu.obs[['cell']].to_csv(fn, sep = '\t', header = False, index = False)
else:
    barcodes = pd.read_csv(simu_barcode_fn, sep = '\t', header = None)
    barcodes = barcodes[0].to_numpy()
    adata_simu.obs["cell"] = barcodes
adata_simu

AnnData object with n_obs × n_vars = 1200 × 21215
    obs: 'cell_type', 'cell'
    var: 'feature'

In [41]:
adata.var['feature'].index = adata.var['feature'].index.astype(str)
assert np.all(adata_simu.var["feature"] == adata.var["feature"])
adata_simu.var = adata.var

In [42]:
fn = os.path.join(work_dir, 'matrix/%s.sccnasim.h5ad' % sample_prefix)
adata_simu.write_h5ad(fn, compression = 'gzip')

In [43]:
fn = os.path.join(out_dir, 'cell_anno.simu.tsv')
adata_simu.obs[['cell', 'cell_type']].to_csv(fn, sep = '\t', header = False, index = False)

Double check the `cell` and `cell_type` of the two matrix are matched.

In [44]:
fn = os.path.join(out_dir, '%s.countmatrix.sccnasim.CellTypeLabel.txt' % sample_prefix)
adata_simu.obs[['cell_type']].to_csv(fn, sep = '\t', header = False, index = False)

### Save to scReadSim count matrix format file

In [45]:
df = pd.DataFrame(adata_simu.X.T.toarray())
df.index = adata_simu.var['feature']
df

,0,1,2,3,4,5,6,7,8,9,...,1190,1191,1192,1193,1194,1195,1196,1197,1198,1199
feature,,,,,,,,,,,,,,,,,,,,,
1_0_29554,0,0,0,0,0,0,1,0,2,0,...,0,0,0,1,2,1,0,3,0,0
1_31109_34554,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1_36081_65419,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1_71585_89295,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1_133723_139790,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9_137217009_137217452,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
9_137219361_137220247,0,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
9_137221581_137224635,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
fn = os.path.join(work_dir, 'matrix/%s.countmatrix.sccnasim.txt' % sample_prefix)
df.to_csv(fn, sep = '\t', header = False, index = True)